# 1. Import Libraries

In [62]:
import os
import sys
import json
import numpy as np
import tensorflow as tf

# 2. File Paths

In [63]:
scripts_path = os.path.abspath(os.path.join('..', 'Scripts'))
if scripts_path not in sys.path:
    sys.path.append(scripts_path)

data_dir = os.path.abspath(os.path.join('..', 'Data', 'TrainTest'))
save_dir = os.path.abspath(os.path.join('..', 'Data', 'SavedModels'))
hyperparameters_dir = os.path.abspath(os.path.join('..', 'Data', 'HyperParameters'))

# 3. Load Models & Data

## 1. Data

In [64]:
train_data = np.load(os.path.join(data_dir, 'train_data.npy')).astype(np.float32)
test_data = np.load(os.path.join(data_dir, 'test_data.npy')).astype(np.float32)
num_items = train_data.shape[1]

## 2. Models

In [65]:
from model import Encoder, Decoder, VAE

# 4. Construct Model

## 1. VAE

In [66]:
vae_progress_file = os.path.join(hyperparameters_dir, 'tuning_progress_vae.json')
with open(vae_progress_file, 'r') as f:
    best_vae_params = json.load(f)['best_params']

# Membangun Arsitektur VAE
encoder = Encoder(hidden_dims=best_vae_params['hidden_dims'], latent_dim=best_vae_params['latent_dim'], dropout_rate=best_vae_params['dropout_rate'])
decoder = Decoder(hidden_dims=best_vae_params['hidden_dims'][::-1], output_dim=num_items)
eval_vae = VAE(encoder, decoder)

# Menyuntikkan Bobot ke VAE
_ = eval_vae(train_data[:1]) 
eval_vae.load_weights(os.path.join(save_dir, 'trained_best_vae_weights.weights.h5'))

## 2. RSVD
    Only matrix item (V)

In [67]:
V_final = np.load(os.path.join(save_dir, 'best_V.npy'))

# 5. Prediction

## 1. Latent Matrix

In [68]:
Z_mean, _ = eval_vae.encoder.predict(train_data, batch_size=best_vae_params['batch_size'])

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


## 2. Filtering with RSVD
    Clean noise with RSVD

In [69]:
Z_projected = np.dot(Z_mean, V_final)         # Reduksi dimensi / Denoising
Z_clean = np.dot(Z_projected, V_final.T)      # Rekonstruksi kembali ke ukuran laten awal

## 3. Decode Latent Matrix
    Decode latent matrix with VAE decoder

In [70]:
predicted_ratings_normalized = eval_vae.decoder.predict(Z_clean, batch_size=best_vae_params['batch_size'])

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


## 4. Denormalize Ratings

In [71]:
MAX_RATING = 5.0 
predicted_ratings = predicted_ratings_normalized * MAX_RATING
test_data_denormalized = test_data * MAX_RATING

# 6. Evaluation

## 1. Masking Data

In [72]:
mask = test_data_denormalized > 0

actual_values = test_data_denormalized[mask]
predicted_values = predicted_ratings[mask]

## 2. Evaluation Metrics Calculation

In [ ]:
mse = np.mean(np.square(actual_values - predicted_values))
rmse = np.sqrt(mse)
mae = np.mean(np.abs(actual_values - predicted_values))

print(f"Mean Squared Error (MSE) : {mse:.4f}")
print(f"Root Mean Squared Error (RMSE) : {rmse:.4f}")
print(f"Mean Absolute Error (MAE)      : {mae:.4f}")

Mean Squared Error (MSE) : 10.5574
Root Mean Squared Error (RMSE) : 3.2492
Mean Absolute Error (MAE)      : 2.9003
